### *Import the Dataset from kaggle - One time thing*

In [11]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "mdismielhossenabir/sentiment-analysis",
  "sentiment_analysis.csv",
  pandas_kwargs={"usecols": ["text", "sentiment"]},
)

df.head()

/var/folders/0n/5dlnlm9j55d7_9bwv3kx8fkc0000gn/T/ipykernel_2754/1988013530.py:5: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


,text,sentiment
0,What a great day!!! Looks like dream.,positive
1,"I feel sorry, I miss you here in the sea beach",positive
2,Don't angry me,negative
3,We attend in the class just for listening teac...,negative
4,"Those who want to go, let them go",negative


### *Download stuff from nltk - One time thing*

In [19]:
import nltk
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package stopwords to /Users/krish/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/krish/nltk_data...


True

# Core Work

## Clean the Dataset

In [32]:
df["sentiment"] = df["sentiment"].str.strip().str.lower()
df["sentiment_encoded"] = df["sentiment"].map({
    "positive": 1,
    "negative": -1
})
df = df.dropna(subset=["sentiment_encoded"])
df["sentiment_encoded"] = df["sentiment_encoded"].astype(int)
df.head()

,text,sentiment,sentiment_encoded
0,great day look like dream,positive,1
1,feel sorry miss sea beach,positive,1
2,dont angry,negative,-1
3,attend class listening teacher reading slide n...,negative,-1
4,want go let go,negative,-1


In [ ]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

STOP_WORDS = set(stopwords.words('english'))

def preprocess_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove punctuation and special characters
    text = re.sub(r'[^\w\s]', '', text)
    
    # Tokenize the text
    tokens = text.split()
    
    # Remove stop words
    tokens = [word for word in tokens if word not in STOP_WORDS]
    
    # Lemmatize the tokens
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    
    return ' '.join(tokens)

In [20]:
df["text"] = df["text"].apply(preprocess_text)

In [16]:
def encode_using_count_vectorizer(texts):
    from sklearn.feature_extraction.text import CountVectorizer
    vectorizer = CountVectorizer()
    return vectorizer.fit_transform(texts)

In [17]:
def encode_using_tfidf_vectorizer(texts):
    from sklearn.feature_extraction.text import TfidfVectorizer
    vectorizer = TfidfVectorizer()
    return vectorizer.fit_transform(texts)

In [40]:
def encode_using_word2vec(texts):
    from gensim.models import Word2Vec
    import numpy as np

    tokens = [text.split() for text in texts]
    model = Word2Vec(tokens, vector_size=300, window=5, min_count=1, epochs=20)

    X = []
    for sentence in tokens:
        vectors = [model.wv[word] for word in sentence]
        X.append(np.mean(vectors, axis=0))

    return np.array(X)

In [34]:
def analyse(texts):
    from sklearn.model_selection import train_test_split
    X_train, X_test, y_train, y_test = train_test_split(texts, df['sentiment_encoded'], test_size = 0.20, random_state = 0)

    from sklearn.linear_model import LogisticRegression
    model = LogisticRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    from sklearn.metrics import accuracy_score, classification_report
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred, target_names=["negative", "positive"]))

In [42]:
def main():
    print("Using Count Vectorizer:")
    X_count = encode_using_count_vectorizer(df["text"])
    analyse(X_count)

    print("\nUsing TF-IDF Vectorizer:")
    X_tfidf = encode_using_tfidf_vectorizer(df["text"])
    analyse(X_tfidf)

    print("\nUsing Word2Vec:")
    X_word2vec = encode_using_word2vec(df["text"])
    analyse(X_word2vec)
    
main()

Using Count Vectorizer:
Accuracy: 0.7666666666666667
Classification Report:
               precision    recall  f1-score   support

    negative       0.76      0.64      0.70        25
    positive       0.77      0.86      0.81        35

    accuracy                           0.77        60
   macro avg       0.77      0.75      0.75        60
weighted avg       0.77      0.77      0.76        60


Using TF-IDF Vectorizer:
Accuracy: 0.8166666666666667
Classification Report:
               precision    recall  f1-score   support

    negative       0.89      0.64      0.74        25
    positive       0.79      0.94      0.86        35

    accuracy                           0.82        60
   macro avg       0.84      0.79      0.80        60
weighted avg       0.83      0.82      0.81        60


Using Word2Vec:
Accuracy: 0.5833333333333334
Classification Report:
               precision    recall  f1-score   support

    negative       0.00      0.00      0.00        25
    positiv

/opt/homebrew/Caskroom/miniconda/base/envs/learn-ml/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniconda/base/envs/learn-ml/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniconda/base/envs/learn-ml/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this b